# 🧠 NeuroSynk - Pipeline Oficial de Entrenamiento en TensorFlow con GPU
### Clasificación Continua de Estados Cognitivos y Detección de Estado Neutro / Basal

Este cuaderno oficial entrena la **Red Neuronal Profunda de NeuroSynk** en TensorFlow / Keras y la exporta en formato **TensorFlow.js (`model.json` y `weights.bin`)** para ejecución 100% local y privada en el navegador.

In [ ]:
# 1. Instalar TensorFlow.js Converter para exportación web
!pip install tensorflowjs numpy pandas scikit-learn matplotlib

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import os

print(f"🟢 TensorFlow Version: {tf.__version__}")
print(f"⚡ GPU Disponible: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# 2. Cargar Dataset Maestro Aumentado
# Si estás en Colab, puedes subir tu archivo dataset_ventanas_aumentado.csv
url = 'https://raw.githubusercontent.com/user/neurosynk/main/dataset_ventanas_aumentado.csv' # o archivo local
try:
    df = pd.read_csv('dataset_ventanas_aumentado.csv')
except:
    print("Por favor sube dataset_ventanas_aumentado.csv en el panel izquierdo de archivos de Colab.")

print("Muestras por clase:")
print(df['label'].value_counts())
df.head()

In [ ]:
# 3. Normalización Z-Score y Preparación de Tensores
feature_cols = [
    'ear_mean', 'ear_min', 'yaw_mean', 'yaw_std', 'pitch_mean', 'pitch_std',
    'frown_mean', 'nose_delta_sum', 'gaze_variance_mean', 'shoulder_angle_mean'
]

X = df[feature_cols].values.astype(np.float32)
y = df['label'].values.astype(np.int32)

means = np.mean(X, axis=0)
stds = np.std(X, axis=0)
stds[stds == 0] = 1.0

X_norm = (X - means) / stds
num_classes = len(np.unique(y))
y_one_hot = tf.keras.utils.to_categorical(y, num_classes=num_classes)

print(f"X shape: {X_norm.shape} | y shape: {y_one_hot.shape}")

In [ ]:
# 4. Arquitectura de la Red Neuronal Profunda (TensorFlow / Keras)
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(len(feature_cols),)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.15),
    
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.10),
    
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.004),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# 5. Entrenamiento con Validación Cruzada
history = model.fit(
    X_norm, y_one_hot,
    epochs=80,
    batch_size=16,
    validation_split=0.20,
    shuffle=True
)

# Graficar Curvas de Aprendizaje
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Entrenamiento')
plt.plot(history.history['val_accuracy'], label='Validación')
plt.title('Precisión de la Red Neuronal')
plt.xlabel('Época')
plt.ylabel('Precisión')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.title('Pérdida (Cross-Entropy)')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.legend()
plt.show()

In [ ]:
# 6. Exportar directamente a formato TensorFlow.js (model.json y weights.bin)
import tensorflowjs as tfjs

os.makedirs('neurosynk_model_tfjs', exist_ok=True)
tfjs.converters.save_keras_model(model, 'neurosynk_model_tfjs')

metadata = {
    'featureMeans': means.tolist(),
    'featureStds': stds.tolist(),
    'classNames': ['ESTUDIO BASAL / NEUTRO', 'ENFOQUE PROFUNDO (FLOW)', 'DISTRACCIÓN', 'FATIGA', 'SOBREESTIMULACIÓN', 'AGOBIO POSTURAL'],
    'accuracy': f"{history.history['val_accuracy'][-1]*100:.1f}%",
    'loss': f"{history.history['val_loss'][-1]:.4f}",
    'epochs': 80,
    'samplesCount': len(X)
}

with open('neurosynk_model_tfjs/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("🎉 Modelo exportado con éxito en la carpeta 'neurosynk_model_tfjs'.")
print("Archivos generados: model.json, weights.bin, metadata.json")
!zip -r neurosynk_model_tfjs.zip neurosynk_model_tfjs